In [2]:
pip install langchain_text_splitters

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

d:\sudhendra\learning projects\GraphRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
csv_path = Path("D:/sudhendra/learning projects/GraphRAG/data/apple23_extractedtxt.csv")

In [3]:
df = pd.read_csv(csv_path)

In [4]:
df.head()

,page,text
0,1,UNITED STATES\nSECURITIES AND EXCHANGE COMMISS...
1,2,Indicate by check mark whether the Registrant ...
2,3,Apple Inc.\nForm 10-K\nFor the Fiscal Year End...
3,4,This Annual Report on Form 10-K (“Form 10-K”) ...
4,5,Services\nAdvertising\nThe Company’s advertisi...


In [5]:
df = df.dropna(subset = "text")
df = df[df["text"].str.strip() != ""]
df.reset_index(drop = True, inplace = True)

In [6]:
len(df)

80

In [7]:
documents = []
for _, row in df.iterrows():
    documents.append({
        "page" : int(row["page"]),
        "text" : row["text"]
    })

documents[0]


{'page': 1,
 'text': 'UNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\nFORM 10-K\n(Mark One)\n☒    ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\nFor the fiscal year ended September\xa030, 2023\nor\n☐    TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\nFor the transition period from \xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0 to \xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0.\nCommission File Number: 001-36743\nApple Inc.\n(Exact name of Registrant as specified in its charter)\nCalifornia\n94-2404110\n(State or other jurisdiction\nof incorporation or organization)\n(I.R.S. Employer Identification No.)\nOne Apple Park Way\nCupertino, California\n95014\n(Address of principal executive offices)\n(Zip Code)\n(408) 996-1010\n(Registrant’s telephone number, including area code)\nSecurities registered pursuant to Section 12(b) of the Act:\nTitle of each class\nTrading \nsymbol(s)\n

In [8]:
documents[8]

{'page': 9,
 'text': 'The Company has a large, global business with sales outside the U.S. representing a majority of the Company’s total net sales, \nand the Company believes that it generally benefits from growth in international trade. Substantially all of the Company’s \nmanufacturing is performed in whole or in part by outsourcing partners located primarily in China mainland, India, Japan, South \nKorea, Taiwan and Vietnam. Restrictions on international trade, such as tariffs and other controls on imports or exports of goods, \ntechnology or data, can materially adversely affect the Company’s operations and supply chain and limit the Company’s ability to \noffer and distribute its products and services to customers. The impact can be particularly significant if these restrictive measures \napply to countries and regions where the Company derives a significant portion of its revenues and/or has significant supply \nchain operations. Restrictive measures can require the Company to t

Chunking the Text

In [9]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 250,
    separators = ["\n\n","\n","."," ",""]
)

chunks = []

for doc in documents:
    page_chunk = splitter.split_text(doc["text"])

    for i,chunk in enumerate(page_chunk):
        chunks.append({
            "page" : doc["page"],
            "chunk_id" : f"page_{doc['page']}_chunk_{i}",
            "text" : chunk
            })
        
        
chunks_df = pd.DataFrame(chunks)

chunks_df.head()

,page,chunk_id,text
0,1,page_1_chunk_0,UNITED STATES\nSECURITIES AND EXCHANGE COMMISS...
1,1,page_1_chunk_1,—\nThe Nasdaq Stock Market LLC\n1.375% Notes d...
2,2,page_2_chunk_0,Indicate by check mark whether the Registrant ...
3,2,page_2_chunk_1,any new or revised financial accounting standa...
4,2,page_2_chunk_2,day of the Registrant’s most recently complete...


In [10]:
len(chunks_df)

239

In [11]:
print(chunks_df.loc[0,"text"])

UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☒    ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended September 30, 2023
or
☐    TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from              to             .
Commission File Number: 001-36743
Apple Inc.
(Exact name of Registrant as specified in its charter)
California
94-2404110
(State or other jurisdiction
of incorporation or organization)
(I.R.S. Employer Identification No.)
One Apple Park Way
Cupertino, California
95014
(Address of principal executive offices)
(Zip Code)
(408) 996-1010
(Registrant’s telephone number, including area code)
Securities registered pursuant to Section 12(b) of the Act:
Title of each class
Trading 
symbol(s)
Name of each exchange on which registered
Common Stock, $0.00001 par value per share
AAPL
The Nasdaq Stock Market LLC
1.375% N

In [12]:
chunks_df.sample(10)

,page,chunk_id,text
102,30,page_30_chunk_0,Item 8. \nFinancial Statements and Supplementa...
39,11,page_11_chunk_4,that can unexpectedly interfere with the inten...
41,12,page_12_chunk_0,The Company is exposed to the risk of write-do...
71,18,page_18_chunk_0,Financial Risks\nThe Company expects its quart...
166,57,page_57_chunk_0,PART IV\nItem 15. \nExhibit and Financial Stat...
144,49,page_49_chunk_0,Share-Based Compensation\nThe following table ...
214,71,page_71_chunk_3,"(2) in the case of the 2013 Indenture, the hol..."
32,10,page_10_chunk_2,"and applications.\nThe Company’s business, res..."
220,73,page_73_chunk_2,"holders of debt securities of, in the case of ..."
180,62,page_62_chunk_1,DESCRIPTION OF COMMON STOCK\nThe following is ...


In [13]:
output_chunks = Path("D:/sudhendra/learning projects/GraphRAG/data/apple23chunks.csv")

In [14]:
chunks_df.to_csv(output_chunks, index = False)
output_chunks

WindowsPath('D:/sudhendra/learning projects/GraphRAG/data/apple23chunks.csv')